In [1]:
import os, sys
import numpy as np
import pickle as pk
from random import randint
from itertools import product, chain
import scipy.interpolate as itp
from multiprocessing import Pool, Process

sys.path.append('/home/nishant/lab/MFB/scripts')
sys.path.append('/home/nishant/lab/MFB/steps')
# sys.path.append('/home/nishant/lab/scripts')
from analysis import *
from peaks import *
from misc import *

## Simulation and analysis of one design for showing sequence of events in response to an AP

In [2]:
resultPath = "/media/nishant/data/results/"
dirs = np.array(getDirs(resultPath, sstr='nVDCC'))

print(len(dirs))
tempdirs = [a for a in dirs if 'nVDCC_12_dVDCC_160_nAZ_7' in a]
for i,d in enumerate(tempdirs):
    print(i, d)

1
0 nVDCC_12_dVDCC_160_nAZ_7


## AZ simulation in STEPS

In [3]:
from MFB_model import *
mdl, sim, r = get_MFB_model()

In [4]:
def runAZTrials(CaData, RRPs):
    vesData = []
    resAZs = []
    # print(CaData[0], CaData[1], RRPs)
    for RRP in RRPs:
        # resCa, resAZ, vesRel = simAZ(CaData[0], CaData[1], sim, r, RRP=RRP)
        resAZ, vesRel = simAZ(CaData[0], CaData[1], sim, r, RRP=RRP)
        vesRelTot = np.sum(vesRel, axis=1)
        pks = detect_peaks(vesRelTot, edge='rising', show=False)
        vesData.append(list(CaData[0,pks]))
        # resAZs.append(resAZ)
    
    # return [vesData, resAZs]
    return [vesData, resAZs, vesRel]  ## changed

In [6]:
def getVesRel(dir, RRPs, resultPath, fname='CaConc.dat', trials=2000):
    nAZ = int(getSimInfo(dir, 'nAZ'))
    rrps = len(RRPs)
    
    CaFile = os.path.join(resultPath, dir, fname)
    CaData = np.genfromtxt(CaFile, unpack=True) # in uM
    
    p = Pool()
    vesRelTimes = []
    vesRels = []
    for iCa in tq(range(1,nAZ+1), desc=dir):
    # for iCa in range(1,nAZ+1):
        
        info = product([CaData[(0,iCa),:]], [RRPs]*trials)
        # print(list(info)[0])
        vRelTime = np.array(p.starmap(runAZTrials, info), dtype=object)
        # vRelTime = runAZTrials(CaData[(0,iCa),:], RRPs)
        # print(vRelTime[:,0])

        vesRelTime = vRelTime[:,0]
        vesRelTime = [[vesRelTime[i][j] for i in range(trials)] for j in range(rrps)]
        vesRelTimes.append(vesRelTime)
        #print(vesRelTime[:,1])

        # AZstates = vRelTime[:,1]
        # AZstates = np.mean(AZstates, axis=0)
        #print(AZstates[0])
        #print(np.vstack((CaData[0,],AZstates[0])))
        # for i,rrp in enumerate(RRPs):
        #     fAZ = os.path.join(resultPath, dir, f'AZ_{iCa}_RRP_{rrp}.dat')
        #     AZdata = np.hstack((np.array([CaData[0]]).T, AZstates[i]))
        #     np.savetxt(fAZ, AZdata, fmt=['%0.5f']+['%0.3f']*18, delimiter="\t")

        vesRel = vRelTime[:,2]  ## changed
        vesRels.append(vesRel)  ## changed

    p.close()
    p.join()
    
    vesRelTimes = [[list(chain(*[vesRelTimes[i][j][k] for i in range(nAZ)])) for k in range(trials)] for j in range(rrps)]
    vesData = {}
    for RRP,v in zip(RRPs, vesRelTimes):
        vesData.update({str(RRP): v})
    
    return vesData, vesRels

dir = tempdirs[0] 
rrp = 30 #int(getSimInfo(dir, 'RRP'))
vesData, vesRels = getVesRel(dir, RRPs=[rrp], resultPath=resultPath, trials=5000)

# with open(os.path.join(resultPath, dirs[0], 'newVesData.dat'), "wb") as outfile:
#     pk.dump(vesData, outfile)
# with open(os.path.join(resultPath, dirs[0], 'newVesRels.dat'), "wb") as outfile:
#     pk.dump(vesRels, outfile)

nVDCC_12_dVDCC_160_nAZ_7: 100%|██████████| 7/7 [02:58<00:00, 25.47s/it]


In [18]:
CaFile = os.path.join(resultPath, dir, 'CaConc.dat')
CaData = np.genfromtxt(CaFile, unpack=True) # in uM

In [ ]:
syncRel = []
asyncRel = []
spontRel = []

for naz in tq(vesRels):
    for trials in naz:
        # print(trials.shape)
        pks = detect_peaks(np.sum(trials[:,:3], axis=1), edge='rising', show=False)
        syncRel.append(list(CaData[0,pks]))

        pks = detect_peaks(np.sum(trials[:,3:9], axis=1), edge='rising', show=False)
        asyncRel.append(list(CaData[0,pks]))

        pks = detect_peaks(trials[:,9], edge='rising', show=False)
        spontRel.append(list(CaData[0,pks]))


syncRel = np.sort(np.concatenate(syncRel))
asyncRel = np.sort(np.concatenate(asyncRel))
spontRel = np.sort(np.concatenate(spontRel))

# syncRel.shape, asyncRel.shape, spontRel.shape
vesRel = {
    'sync': syncRel, 
    'async': asyncRel, 
    'spont': spontRel
}

with open(os.path.join(resultPath, dirs[0], 'newVesRel.dat'), "wb") as outfile:
    pk.dump(vesRel, outfile)

syncRel, asyncRel, spontRel